In [6]:
batch_size = 24
batches_count = 5
# endpoint = "http://localhost:3000/generate_anime"
endpoint = "http://192.69.210.82:80/generate_anime"

In [7]:
import asyncio
from dataclasses import dataclass
import time
import httpx

from model import GenerateRequest, PipelineParams

@dataclass
class OperationResult:
    elapsed_time: float
    is_success: bool
    
def generate_request(prompt: str, seed: int) -> GenerateRequest:
    return GenerateRequest(
        prompt=prompt,
        seed=seed,
        pipeline_params=PipelineParams(
            num_inference_steps=25,
            width=1024,
            height=1024,
            guidance_scale=7.0,
            negative_prompt="out of frame, nude, duplicate, watermark, signature, mutated, text, blurry, worst quality, low quality, artificial, texture artifacts, jpeg artifacts"
        ),
        timeout=12
    )

async def send_request(request: GenerateRequest) -> OperationResult:
    time_start = time.time()
    async with httpx.AsyncClient(timeout=20) as client:
        response = await client.post(endpoint, json=request.model_dump())
    elapsed_time = time.time() - time_start
    
    print(f"Request took {elapsed_time} seconds and returned status code {response.status_code}")
    
    return OperationResult(elapsed_time=elapsed_time, is_success=response.status_code == 200)

async def send_batch(requests: list[GenerateRequest]) -> list[OperationResult]:
    tasks = [send_request(request) for request in requests]
    return await asyncio.gather(*tasks)

async def benchmark():
    time_start = time.time()
    requests = [generate_request("hyper-realistic photo, the living room has black furniture and beige walls, aesthetic, shot on Sony --style raw", i) for i in range(batch_size * batches_count)]
    batches = [requests[i:i+batch_size] for i in range(0, len(requests), batch_size)]
    results = []
    
    for batch in batches:
        results.extend(await send_batch(batch))
        
    return { "results": results, "elapsed_time": time.time() - time_start }

results = await benchmark()

Request took 1.6080880165100098 seconds and returned status code 200
Request took 1.7028772830963135 seconds and returned status code 200
Request took 2.499284505844116 seconds and returned status code 200
Request took 2.4807980060577393 seconds and returned status code 200
Request took 3.365926742553711 seconds and returned status code 200
Request took 3.402606248855591 seconds and returned status code 200
Request took 4.696851491928101 seconds and returned status code 200
Request took 4.81541109085083 seconds and returned status code 200
Request took 5.410724401473999 seconds and returned status code 200
Request took 5.379353761672974 seconds and returned status code 200
Request took 6.740090847015381 seconds and returned status code 200
Request took 6.840176820755005 seconds and returned status code 200
Request took 7.81683349609375 seconds and returned status code 200
Request took 7.945906639099121 seconds and returned status code 200
Request took 8.659389972686768 seconds and retu

In [8]:
def generate_report(results: list[OperationResult], elapsed_time: float):
    print(f"{len(results)} requests were sent")
    print(f"{len([r for r in results if r.is_success])} requests were successful")
    print(f"{len([r for r in results if not r.is_success])} requests failed")
    
    average_time = sum([r.elapsed_time for r in results if r.is_success]) / len([r for r in results if r.is_success])
    print(f"Average response time: {average_time} seconds")
    
    requests_per_second = len([r for r in results if r.is_success]) / elapsed_time
    print(f"Requests per second: {requests_per_second}")
    print(f"Total Volume: {int(requests_per_second * 600)}")

generate_report(results["results"], results["elapsed_time"])

120 requests were sent
110 requests were successful
10 requests failed
Average response time: 6.720177730647 seconds
Requests per second: 1.777885859731073
Total Volume: 1066
